In [1]:
"""
HomeGuard Security System Simulator
Author: Tamara
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.
"""

import random
from datetime import datetime
import time

# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

# Current system state
current_mode = "AWAY"

def create_sensor(sensor_id, location, sensor_type, threshold=None):
    """Creates a sensor data structure (dictionary)."""
    sensor = {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "threshold": threshold,
        "current_value": None
    }
    return sensor

def create_alert(severity, message, sensor_id, timestamp):
    """Creates an alert data structure (dictionary)."""
    alert = {
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp
    }
    return alert

In [2]:
def is_abnormal_reading(sensor, reading_value):
    """Checks if a sensor reading is abnormal based on sensor type and thresholds."""
    sensor_type = sensor["type"]
    
    if sensor_type == "temperature":
        if reading_value < 35 or reading_value > 95:
            return True
    elif sensor_type == "motion":
        if reading_value == True:
            return True
    elif sensor_type == "door":
        if reading_value == "OPENED":
            return True
    elif sensor_type == "smoke":
        if reading_value == "DETECTED":
            return True
            
    return False

def process_reading(sensor, reading_value, system_mode):
    """Processes a sensor reading and determines if alerts are needed."""
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    
    # 1. Security alerts
    if system_mode == "AWAY":
        if sensor["type"] == "motion" and reading_value == True:
            alerts.append(create_alert("HIGH", f"SECURITY: Motion detected in {sensor['location']} while in AWAY mode!", sensor["id"], timestamp))
        if sensor["type"] == "door" and reading_value == "OPENED":
            alerts.append(create_alert("HIGH", f"SECURITY: {sensor['location']} opened while in AWAY mode!", sensor["id"], timestamp))
                    
    # 2. Safety alerts
    if sensor["type"] == "temperature":
        if reading_value < 35:
            alerts.append(create_alert("CRITICAL", f"SAFETY: Freezing temperature risk in {sensor['location']} ({reading_value}°F)!", sensor["id"], timestamp))
        elif reading_value > 95:
            alerts.append(create_alert("CRITICAL", f"SAFETY: Equipment failure/overheat risk in {sensor['location']} ({reading_value}°F)!", sensor["id"], timestamp))
                    
    if sensor["type"] == "smoke" and reading_value == "DETECTED":
        alerts.append(create_alert("CRITICAL", f"SAFETY: Smoke detected in {sensor['location']}! Fire risk!", sensor["id"], timestamp))
                
    # 3. Comfort notifications
    if system_mode == "HOME" and sensor["type"] == "temperature":
        if reading_value < 65 or reading_value > 75:
            alerts.append(create_alert("LOW", f"COMFORT: Temperature in {sensor['location']} is outside comfort range ({reading_value}°F).", sensor["id"], timestamp))

    return alerts

def generate_reading(sensor_type):
    """Generates a realistic reading for a sensor type."""
    if sensor_type == "temperature":
        return random.randint(32, 85)
    elif sensor_type == "motion":
        return random.choice([True, False, False, False])
    elif sensor_type == "door":
        return random.choice(["CLOSED", "CLOSED", "CLOSED", "OPENED"])
    elif sensor_type == "smoke":
        return random.choice(["CLEAR", "CLEAR", "CLEAR", "DETECTED"])
    return None

def trigger_alert(alert):
    """Displays an alert to the user."""
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")

def log_event(message, timestamp=None):
    """Logs an event to the console."""
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [3]:
class Sensor:
    """Represents a sensor in the HomeGuard system using Object-Oriented Programming."""
    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None
    
    def read(self):
        self.current_value = generate_reading(self.type)
        return self.current_value
        
    def isAbnormal(self):
        sensor_dict = {
            "id": self.id,
            "location": self.location,
            "type": self.type,
            "threshold": self.threshold
        }
        return is_abnormal_reading(sensor_dict, self.current_value)
    
    def reset(self):
        self.current_value = None
    
    def __str__(self):
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"

In [4]:
def run_simulation(duration_minutes=3, system_mode="AWAY"):
    """Runs the HomeGuard security system simulation."""
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]
    
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")
        
        for sensor in sensors:
            reading = sensor.read()
            
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")
        
        time.sleep(0.5)
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)

# Run de simulatie
run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 14:59:38
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 34°F (Abnormal)
[ALERT!] 🔥 CRITICAL: SAFETY: Freezing temperature risk in Kitchen (34°F)!
[LOG] [14:59:38] Sending notification to homeowner...
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 14:59:39
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 38°F (Abnormal)
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:59:39] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 14:59:39
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 42°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Simulation complete!
